# 02 — Data Setup, Flat Processed Dataset, and Split Files

This notebook replaces the old `data_download.ipynb` behavior.

It does **not download** datasets. It verifies raw files, rebuilds the processed manifest layer, copies local images into one flat processed image folder, and writes split ID files.

Outputs:

```text
data/processed/ooc_pairs/images/
data/processed/ooc_pairs/manifests/*.csv
data/processed/ooc_pairs/reports/manifest_summary.csv
data/splits/*.txt
```

Run order:

```text
1. env_check_updated.ipynb
2. data_setup_updated.ipynb
3. eda_updated.ipynb
```


In [1]:
from pathlib import Path
import sys
import json
import shutil
import re
import pandas as pd

def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()

    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "src").exists():
            return p

    raise RuntimeError(
        f"Could not find repo root from {start}. "
        "Open Jupyter from E:/eve-rag or move this notebook inside the repo."
    )

ROOT = find_repo_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Detected ROOT:", ROOT)
print("src exists:", (ROOT / "src").exists())
print("paths.py exists:", (ROOT / "src" / "utils" / "paths.py").exists())

from src.utils.paths import (
    DATA,
    RAW,
    PROCESSED,
    MANIFESTS,
    PROCESSED_IMAGES,
    REPORTS,
    SPLITS,
    NEWSCLIPPINGS_RAW,
    MMFAKEBENCH_RAW,
    VERITE_CANON,
    VERITE_CSV,
    VERITE_IMAGES,
)

IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff", ".gif"]

for p in [PROCESSED_IMAGES, MANIFESTS, REPORTS, SPLITS]:
    p.mkdir(parents=True, exist_ok=True)

print("MANIFESTS:", MANIFESTS)
print("PROCESSED_IMAGES:", PROCESSED_IMAGES)
print("SPLITS:", SPLITS)

Detected ROOT: E:\eve-rag
src exists: True
paths.py exists: True
MANIFESTS: E:\eve-rag\data\processed\ooc_pairs\manifests
PROCESSED_IMAGES: E:\eve-rag\data\processed\ooc_pairs\images
SPLITS: E:\eve-rag\data\splits


In [2]:
# Rebuild controls
# True  = delete old processed images/manifests/reports and rebuild cleanly
# False = keep existing processed files and only overwrite manifests
REBUILD_FROM_SCRATCH = True

print("REBUILD_FROM_SCRATCH:", REBUILD_FROM_SCRATCH)

REBUILD_FROM_SCRATCH: True


## 1. Clean and create processed folders


In [3]:
for folder in [PROCESSED_IMAGES, MANIFESTS, REPORTS, SPLITS]:
    folder.mkdir(parents=True, exist_ok=True)

if REBUILD_FROM_SCRATCH:
    for folder in [PROCESSED_IMAGES, MANIFESTS, REPORTS]:
        if folder.exists():
            for item in folder.iterdir():
                if item.is_dir():
                    shutil.rmtree(item)
                else:
                    item.unlink()
        folder.mkdir(parents=True, exist_ok=True)
    print("Cleared processed images, manifests, and reports.")
else:
    print("Keeping existing processed files.")


Cleared processed images, manifests, and reports.


## 2. Helper functions


In [4]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_records(path: Path):
    obj = load_json(path)
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        if "annotations" in obj:
            ann = obj["annotations"]
            if isinstance(ann, list):
                return ann
            if isinstance(ann, dict):
                return list(ann.values())
        for key in ["data", "items", "samples", "examples", "pairs"]:
            if key in obj:
                val = obj[key]
                if isinstance(val, list):
                    return val
                if isinstance(val, dict):
                    return list(val.values())
        if all(isinstance(v, dict) for v in obj.values()):
            rows = []
            for k, v in obj.items():
                row = dict(v)
                row.setdefault("_key", k)
                rows.append(row)
            return rows
        if all(isinstance(v, list) for v in obj.values()):
            return pd.DataFrame(obj).to_dict("records")
    raise ValueError(f"Unsupported JSON structure: {path}")


def safe_str(x):
    if x is None:
        return ""
    if not isinstance(x, (list, dict)):
        try:
            if pd.isna(x):
                return ""
        except Exception:
            pass
    return str(x)


def norm_label(value):
    s = safe_str(value).strip().lower()
    if s in {"0", "true", "real", "matched", "match", "pristine", "truthful", "yes", "original"}:
        return 0
    if s in {"1", "false", "fake", "falsified", "ooc", "out-of-context", "mismatch", "miscaptioned", "no"}:
        return 1
    return None


def label_name(label):
    if label == 0:
        return "match"
    if label == 1:
        return "mismatch"
    return "unknown"


def relpath(p: Path):
    if p is None:
        return ""
    return str(p.relative_to(ROOT)).replace("\\", "/")


def sanitize_token(s: str):
    s = safe_str(s).strip().replace("\\", "/").replace(" ", "_")
    s = Path(s).name if "/" in s else s
    s = re.sub(r"[^A-Za-z0-9._-]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "item"


def find_by_name_or_stem(folder: Path, value):
    s = safe_str(value).strip()
    if not s or s.lower() == "nan":
        return None
    name = Path(s).name
    stem = Path(name).stem
    candidates = [folder / name]
    for ext in IMAGE_EXTS:
        candidates.append(folder / f"{stem}{ext}")
    try:
        n = int(float(stem))
        for width in [1, 4, 5, 6]:
            for ext in IMAGE_EXTS:
                candidates.append(folder / f"{n:0{width}d}{ext}")
    except Exception:
        pass
    for p in candidates:
        if p.exists():
            return p
    return None


def copy_image_flat(src: Path, unique_stem: str):
    if src is None or not src.exists():
        return None
    ext = src.suffix.lower()
    if ext not in IMAGE_EXTS:
        return None
    dst = PROCESSED_IMAGES / f"{unique_stem}{ext}"
    if not dst.exists():
        shutil.copy2(src, dst)
    return dst


BASE_COLUMNS = [
    "pair_id", "dataset", "split",
    "image_id", "image_path", "raw_image_path", "has_image",
    "caption", "label", "label_name",
    "source_image_id", "source_text_id", "original_id",
    "clip_image_key", "clip_text_key",
    "text_source", "image_source", "fake_cls",
    "topic", "source", "timestamp", "title",
    "raw_label", "notes",
]


def finalize(df: pd.DataFrame):
    for col in BASE_COLUMNS:
        if col not in df.columns:
            df[col] = ""
    df = df[BASE_COLUMNS].copy()
    for col in df.columns:
        if col not in {"label", "has_image"}:
            df[col] = df[col].fillna("").astype(str)
    df["label"] = pd.to_numeric(df["label"], errors="coerce").astype("Int64")
    df["has_image"] = pd.to_numeric(df["has_image"], errors="coerce").fillna(0).astype(int)
    return df

print("Helpers loaded.")


Helpers loaded.


## 3. Verify raw datasets


In [5]:
required = [
    NEWSCLIPPINGS_RAW / "train.json",
    NEWSCLIPPINGS_RAW / "val.json",
    NEWSCLIPPINGS_RAW / "test.json",
    NEWSCLIPPINGS_RAW / "metadata" / "train.json",
    NEWSCLIPPINGS_RAW / "metadata" / "val.json",
    NEWSCLIPPINGS_RAW / "metadata" / "test.json",
    VERITE_CSV,
    VERITE_IMAGES,
    MMFAKEBENCH_RAW / "MMFakeBench_val.json",
    MMFAKEBENCH_RAW / "MMFakeBench_test.json",
]

missing = []
for p in required:
    if p.exists():
        print("✓", p.relative_to(ROOT))
    else:
        print("✗", p)
        missing.append(p)

if missing:
    raise FileNotFoundError(f"Missing {len(missing)} required raw files/folders. See printed paths.")


✓ data\raw\newsclippings\train.json
✓ data\raw\newsclippings\val.json
✓ data\raw\newsclippings\test.json
✓ data\raw\newsclippings\metadata\train.json
✓ data\raw\newsclippings\metadata\val.json
✓ data\raw\newsclippings\metadata\test.json
✓ data\raw\verite\image-text-verification\VERITE\VERITE.csv
✓ data\raw\verite\image-text-verification\VERITE\images
✓ data\raw\mmfakebench\MMFakeBench_val.json
✓ data\raw\mmfakebench\MMFakeBench_test.json


## 4. Build NewsCLIPpings manifests


In [6]:
def build_news_metadata(split_name):
    meta_path = NEWSCLIPPINGS_RAW / "metadata" / f"{split_name}.json"
    raw = load_json(meta_path)
    index = {}
    iterator = raw.items() if isinstance(raw, dict) else [(None, row) for row in load_records(meta_path)]
    for key, row in iterator:
        if not isinstance(row, dict):
            continue
        keys = {
            safe_str(key),
            safe_str(row.get("id", "")),
            safe_str(row.get("image_id", "")),
            Path(safe_str(row.get("image_path", ""))).name,
            Path(safe_str(row.get("image_path", ""))).stem,
        }
        for k in keys:
            if k and k != "nan":
                index[k] = row
                try:
                    index[str(int(float(k)))] = row
                except Exception:
                    pass
    return index


def build_news(split_name):
    pairs_path = NEWSCLIPPINGS_RAW / f"{split_name}.json"
    pairs = load_records(pairs_path)
    meta = build_news_metadata(split_name)
    rows = []
    for idx, r in enumerate(pairs):
        pair_id_raw = r.get("id", idx)
        image_id_raw = r.get("image_id", r.get("image", r.get("image_path", "")))
        y = int(r.get("falsified", r.get("label", 0)))
        m = meta.get(safe_str(image_id_raw)) or meta.get(safe_str(pair_id_raw))
        caption = raw_image_path = topic = source = timestamp = title = ""
        if m:
            caption = safe_str(m.get("caption", ""))
            raw_image_path = safe_str(m.get("image_path", ""))
            topic = safe_str(m.get("topic", ""))
            source = safe_str(m.get("source", ""))
            timestamp = safe_str(m.get("timestamp", ""))
            title = safe_str(m.get("title", ""))
        rows.append({
            "pair_id": f"nc_{pair_id_raw}_{y}",
            "dataset": "newsclippings",
            "split": split_name,
            "image_id": f"ncimg_{sanitize_token(image_id_raw)}",
            "image_path": "",
            "raw_image_path": raw_image_path,
            "has_image": 0,
            "caption": caption,
            "label": y,
            "label_name": label_name(y),
            "source_image_id": image_id_raw,
            "source_text_id": pair_id_raw,
            "original_id": pair_id_raw,
            "clip_image_key": image_id_raw,
            "clip_text_key": pair_id_raw,
            "topic": topic,
            "source": source,
            "timestamp": timestamp,
            "title": title,
            "raw_label": y,
            "notes": "embedding-only row; local VisualNews image unavailable",
        })
    out = finalize(pd.DataFrame(rows))
    out.to_csv(MANIFESTS / {"train":"train.csv","val":"val.csv","test":"test.csv"}[split_name], index=False, encoding="utf-8")
    return out


## 5. Build VERITE manifest and flat images


In [7]:
def build_verite():
    df = pd.read_csv(VERITE_CSV)
    rows = []
    copied = 0
    for i, r in df.iterrows():
        raw_image_path = safe_str(r.get("image_path", "")).strip()
        caption = safe_str(r.get("caption", r.get("text", ""))).strip()
        raw_label = r.get("label", "")
        y = norm_label(raw_label)
        candidates = []
        p = find_by_name_or_stem(VERITE_IMAGES, raw_image_path)
        if p:
            candidates.append(p)
        for stem in [str(i), f"{i:04d}", f"{i+1:04d}", f"{i:05d}", f"{i+1:05d}"]:
            p = find_by_name_or_stem(VERITE_IMAGES, stem)
            if p:
                candidates.append(p)
        found = candidates[0] if candidates else None
        processed_img = None
        image_id = ""
        if found:
            unique_stem = f"verite_{sanitize_token(found.stem)}"
            processed_img = copy_image_flat(found, unique_stem)
            image_id = unique_stem
            if processed_img is not None:
                copied += 1
        rows.append({
            "pair_id": f"verite_{i:04d}",
            "dataset": "verite",
            "split": "test",
            "image_id": image_id,
            "image_path": relpath(processed_img) if processed_img else "",
            "raw_image_path": raw_image_path,
            "has_image": int(processed_img is not None),
            "caption": caption,
            "label": y,
            "label_name": label_name(y),
            "source_image_id": Path(raw_image_path).stem,
            "source_text_id": safe_str(r.get("Unnamed: 0", i)),
            "original_id": safe_str(r.get("Unnamed: 0", i)),
            "clip_image_key": safe_str(r.get("Unnamed: 0", i)),
            "clip_text_key": safe_str(r.get("Unnamed: 0", i)),
            "raw_label": raw_label,
            "notes": "copied to flat processed image folder" if processed_img else "missing local image; row kept for accounting",
        })
    out = finalize(pd.DataFrame(rows))
    out.to_csv(MANIFESTS / "verite_test.csv", index=False, encoding="utf-8")
    print("VERITE copied/resolved image rows:", copied)
    return out


## 6. Build MMFakeBench manifests and flat images


In [8]:
def resolve_mmfake_image(img_rel, split_name):
    s = safe_str(img_rel).strip()
    if not s or s.lower() == "nan":
        return None
    s_clean = s.replace("\\", "/").lstrip("/")
    name = Path(s_clean).name
    candidates = [
        MMFAKEBENCH_RAW / s_clean,
        MMFAKEBENCH_RAW / f"MMFakeBench_{split_name}" / s_clean,
        MMFAKEBENCH_RAW / f"MMFakeBench_{split_name}" / name,
    ]
    for p in candidates:
        if p.exists():
            return p
    split_dir = MMFAKEBENCH_RAW / f"MMFakeBench_{split_name}"
    if split_dir.exists() and name:
        matches = list(split_dir.rglob(name))
        if matches:
            return matches[0]
    return None


def build_mmfake(split_name):
    json_path = MMFAKEBENCH_RAW / f"MMFakeBench_{split_name}.json"
    records = load_records(json_path)
    rows = []
    copied = 0
    for i, r in enumerate(records):
        caption = safe_str(r.get("text", r.get("caption", r.get("claim", "")))).strip()
        raw_image_path = safe_str(r.get("image_path", r.get("image", r.get("img_path", "")))).strip()
        raw_label = r.get("gt_answers", r.get("label", ""))
        y = norm_label(raw_label)
        if y is None:
            y = 1 if "/fake/" in raw_image_path.replace("\\", "/").lower() else 0
        raw_img = resolve_mmfake_image(raw_image_path, split_name)
        processed_img = None
        image_id = ""
        if raw_img:
            s_clean = raw_image_path.replace("\\", "/").lstrip("/")
            rel_token = sanitize_token(s_clean.replace("/", "_"))
            unique_stem = f"mmfakebench_{split_name}_{Path(rel_token).stem}"
            processed_img = copy_image_flat(raw_img, unique_stem)
            image_id = unique_stem
            if processed_img is not None:
                copied += 1
        rows.append({
            "pair_id": f"mmfakebench_{split_name}_{i:05d}",
            "dataset": "mmfakebench",
            "split": split_name,
            "image_id": image_id,
            "image_path": relpath(processed_img) if processed_img else "",
            "raw_image_path": raw_image_path,
            "has_image": int(processed_img is not None),
            "caption": caption,
            "label": y,
            "label_name": label_name(y),
            "source_image_id": Path(raw_image_path).stem,
            "source_text_id": i,
            "original_id": i,
            "text_source": safe_str(r.get("text_source", "")),
            "image_source": safe_str(r.get("image_source", "")),
            "fake_cls": safe_str(r.get("fake_cls", "")),
            "raw_label": raw_label,
            "notes": "copied to flat processed image folder" if processed_img else "missing local image",
        })
    out = finalize(pd.DataFrame(rows))
    out.to_csv(MANIFESTS / f"mmfakebench_{split_name}.csv", index=False, encoding="utf-8")
    print(f"MMFakeBench {split_name} copied/resolved image rows:", copied)
    return out


## 7. Build all manifests


In [9]:
outputs = []

print("Building NewsCLIPpings...")
for split in ["train", "val", "test"]:
    df = build_news(split)
    outputs.append(df)
    print(f"  {split}: rows={len(df)}, captions={(df['caption'].str.len() > 0).sum()}, has_image={df['has_image'].sum()}")

print("Building VERITE...")
verite_df = build_verite()
outputs.append(verite_df)
print(f"  verite_test: rows={len(verite_df)}, has_image={verite_df['has_image'].sum()}")

print("Building MMFakeBench...")
for split in ["val", "test"]:
    df = build_mmfake(split)
    outputs.append(df)
    print(f"  mmfakebench_{split}: rows={len(df)}, has_image={df['has_image'].sum()}")

all_df = finalize(pd.concat(outputs, ignore_index=True))
all_df.to_csv(MANIFESTS / "all.csv", index=False, encoding="utf-8")

print("\nSaved all.csv:", len(all_df))
print("Wrote manifests to:", MANIFESTS)
print("Wrote flat processed images to:", PROCESSED_IMAGES)


Building NewsCLIPpings...
  train: rows=71072, captions=71072, has_image=0
  val: rows=7024, captions=7024, has_image=0
  test: rows=7264, captions=7264, has_image=0
Building VERITE...
VERITE copied/resolved image rows: 621
  verite_test: rows=1001, has_image=621
Building MMFakeBench...
MMFakeBench val copied/resolved image rows: 1000
  mmfakebench_val: rows=1000, has_image=1000
MMFakeBench test copied/resolved image rows: 10000
  mmfakebench_test: rows=10000, has_image=10000

Saved all.csv: 97361
Wrote manifests to: E:\eve-rag\data\processed\ooc_pairs\manifests
Wrote flat processed images to: E:\eve-rag\data\processed\ooc_pairs\images


## 8. Write split files


In [10]:
split_sources = {
    "train_pair_ids.txt": MANIFESTS / "train.csv",
    "val_pair_ids.txt": MANIFESTS / "val.csv",
    "test_pair_ids.txt": MANIFESTS / "test.csv",
    "verite_test_pair_ids.txt": MANIFESTS / "verite_test.csv",
    "mmfakebench_val_pair_ids.txt": MANIFESTS / "mmfakebench_val.csv",
    "mmfakebench_test_pair_ids.txt": MANIFESTS / "mmfakebench_test.csv",
}

for split_file, manifest_path in split_sources.items():
    df = pd.read_csv(manifest_path, usecols=["pair_id"], dtype={"pair_id": "string"})
    out_path = SPLITS / split_file
    out_path.write_text("\n".join(df["pair_id"].astype(str).tolist()) + "\n", encoding="utf-8")
    print(f"{split_file}: {len(df):,} ids")

all_ids = all_df["pair_id"].astype(str).tolist()
(SPLITS / "all_pair_ids.txt").write_text("\n".join(all_ids) + "\n", encoding="utf-8")
print(f"all_pair_ids.txt: {len(all_ids):,} ids")

local_ids = all_df.loc[all_df["has_image"] == 1, "pair_id"].astype(str).tolist()
(SPLITS / "local_image_pair_ids.txt").write_text("\n".join(local_ids) + "\n", encoding="utf-8")
print(f"local_image_pair_ids.txt: {len(local_ids):,} ids")

news_ids = all_df.loc[all_df["dataset"] == "newsclippings", "pair_id"].astype(str).tolist()
(SPLITS / "newsclippings_pair_ids.txt").write_text("\n".join(news_ids) + "\n", encoding="utf-8")
print(f"newsclippings_pair_ids.txt: {len(news_ids):,} ids")

ood_ids = all_df.loc[all_df["dataset"].isin(["verite", "mmfakebench"]), "pair_id"].astype(str).tolist()
(SPLITS / "ood_pair_ids.txt").write_text("\n".join(ood_ids) + "\n", encoding="utf-8")
print(f"ood_pair_ids.txt: {len(ood_ids):,} ids")


train_pair_ids.txt: 71,072 ids
val_pair_ids.txt: 7,024 ids
test_pair_ids.txt: 7,264 ids
verite_test_pair_ids.txt: 1,001 ids
mmfakebench_val_pair_ids.txt: 1,000 ids
mmfakebench_test_pair_ids.txt: 10,000 ids
all_pair_ids.txt: 97,361 ids
local_image_pair_ids.txt: 11,621 ids
newsclippings_pair_ids.txt: 85,360 ids
ood_pair_ids.txt: 12,001 ids


## 9. Validate manifests and images


In [11]:
summary_rows = []
for csv_name in ["train.csv", "val.csv", "test.csv", "verite_test.csv", "mmfakebench_val.csv", "mmfakebench_test.csv", "all.csv"]:
    path = MANIFESTS / csv_name
    df = pd.read_csv(path, dtype={"image_path":"string", "caption":"string", "pair_id":"string", "image_id":"string"}, low_memory=False)
    broken = df[(df["has_image"] == 1) & (~df["image_path"].fillna("").map(lambda p: (ROOT / p).exists()))]
    summary_rows.append({
        "file": csv_name,
        "rows": len(df),
        "label_0": int((df["label"] == 0).sum()),
        "label_1": int((df["label"] == 1).sum()),
        "has_image": int(df["has_image"].sum()),
        "unique_image_ids": int(df["image_id"].replace("", pd.NA).dropna().nunique()),
        "empty_captions": int((df["caption"].fillna("").astype(str).str.len() == 0).sum()),
        "broken_image_paths": len(broken),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(REPORTS / "manifest_summary.csv", index=False)
summary_df


,file,rows,label_0,label_1,has_image,unique_image_ids,empty_captions,broken_image_paths
0,train.csv,71072,35536,35536,0,59876,0,0
1,val.csv,7024,3512,3512,0,5943,0,0
2,test.csv,7264,3632,3632,0,6147,0,0
3,verite_test.csv,1001,338,663,621,495,0,0
4,mmfakebench_val.csv,1000,300,700,1000,1000,0,0
5,mmfakebench_test.csv,10000,3000,7000,10000,10000,0,0
6,all.csv,97361,46318,51043,11621,83461,0,0


In [12]:
flat_image_count = sum(1 for p in PROCESSED_IMAGES.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
print("Flat processed image count:", flat_image_count)
print("Sample processed filenames:")
for p in sorted([p for p in PROCESSED_IMAGES.iterdir() if p.is_file()])[:20]:
    print(" ", p.name)


Flat processed image count: 11495
Sample processed filenames:
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_1.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_10.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_100.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_101.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_102.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_103.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_104.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_105.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_106.png
  mmfakebench_test_fake_antifact_image_ge